# 1. What you'll learn

        - compare no, complete, and partial pooling
- fit a PyMC binomial hierarchy
- read shrinkage and credible intervals

        **The one question this notebook answers:** How can small groups borrow strength without pretending every group is identical?

# 2. Setup

This lesson keeps bayesian hierarchical modeling small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import os
os.environ.setdefault("PYTENSOR_FLAGS","cxx=")
import pymc as pm
import arviz as az
from scipy.special import expit,logit
from sklearn.metrics import mean_squared_error

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Twelve stores have unequal customer counts and different latent conversion rates. The observed target is conversions out of customers; simulation preserves truth for evaluation.

**Small example:** Rates 1/2 and 450/1000 are both near 0.5, but the first should move much more toward the population because two trials carry little evidence.

In [ ]:
rng=np.random.default_rng(SEED);groups=12;n=np.array([8,12,18,25,35,50,75,110,160,230,350,500]);true_p=expit(rng.normal(logit(.28),.65,groups));success=rng.binomial(n,true_p)
frame=pd.DataFrame({"store":np.arange(groups),"customers":n,"conversions":success,"raw":success/n,"truth":true_p});print("Shape:",frame.shape,"total:",success.sum(),"/",n.sum());print(frame.round(3))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
raw=success/n;pooled=success.sum()/n.sum();se=np.sqrt(np.clip(raw*(1-raw)/n,1e-6,None));fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].bar(range(groups),n,color="#176b66");axes[0].set_title("Unequal evidence")
axes[1].scatter(n,raw,c=true_p,cmap="viridis");axes[1].axhline(pooled,ls="--",color="black");axes[1].set(title="Raw rates",xlabel="customers")
axes[2].scatter(n,se,color="#d38b45");axes[2].set(title="Uncertainty falls with n",xlabel="customers",ylabel="standard error");plt.tight_layout();plt.show()

**Takeaway:** Stores differ more than sixtyfold in evidence.

**Takeaway:** The smallest raw rates are most vulnerable to sampling accidents.

**Takeaway:** A shared shrinkage amount would ignore how uncertainty changes with sample size.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
print(f"Complete-pooling rate={pooled:.3f}; raw range=({raw.min():.3f},{raw.max():.3f})");print("Observed arrays:",success.shape,n.shape)

# 6. Train

        Training turns the assumptions behind bayesian hierarchical modeling into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Place priors on the population logit and group spread.
2. Draw a standardized offset for every group.
3. Transform population plus offset through logistic.
4. Condition on binomial counts and sample the posterior.

In [ ]:
with pm.Model() as hierarchy:
 mu=pm.Normal("mu",logit(.3),1.5);tau=pm.HalfNormal("tau",1);z=pm.Normal("z",0,1,shape=groups);p=pm.Deterministic("p",pm.math.sigmoid(mu+tau*z));pm.Binomial("seen",n=n,p=p,observed=success)
 trace=pm.sample(120,tune=120,chains=2,cores=1,random_seed=SEED,progressbar=False,target_accept=.9,compute_convergence_checks=False)
post=trace.posterior["p"].stack(sample=("chain","draw")).values;post_mean=post.mean(1);hdi=az.hdi(trace.posterior["p"],hdi_prob=.9)["p"].values;print("Draw matrix:",post.shape,"divergences:",int(trace.sample_stats.diverging.sum()))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
rmse=lambda a:mean_squared_error(true_p,a)**.5;print(f"Raw RMSE={rmse(raw):.3f}; pooled={rmse(np.repeat(pooled,groups)):.3f}; partial={rmse(post_mean):.3f}")
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].scatter(raw,post_mean,c=n,cmap="viridis");axes[0].plot([0,1],[0,1],"--",color="black");axes[0].set(title="Shrinkage",xlabel="raw",ylabel="posterior")
axes[1].errorbar(range(groups),post_mean,yerr=[post_mean-hdi[:,0],hdi[:,1]-post_mean],fmt="o");axes[1].scatter(range(groups),true_p,marker="x",color="#d38b45");axes[1].set_title("90% intervals and truth")
axes[2].scatter(n,np.abs(post_mean-raw),color="#d38b45");axes[2].set(title="Small groups move most",xlabel="customers");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Excess pooling can hide real outlying groups; too little pooling leaves tiny groups governed by noise. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
strong=(success+100*pooled)/(n+100);weak=(success+5*pooled)/(n+5)
print(f"Strong pseudo-count RMSE={rmse(strong):.3f}; weak={rmse(weak):.3f}");small=np.argmin(n);print(f"Small store raw={raw[small]:.3f}, partial={post_mean[small]:.3f}, over-pooled={strong[small]:.3f}, truth={true_p[small]:.3f}")

**Use this when…** related groups have unequal evidence and uncertainty matters.

        **Don't use this when…** groups are unrelated, key group covariates are omitted, or posterior diagnostics cannot be checked.

        ## Try this

        1. Give every group 500 trials and watch shrinkage fade.
2. Increase true group spread and see less pooling.
3. Add a store-level covariate to the population logit.